Notebook для практики по дисциплине "Методы машинного обучения"
Выполнили студенты:
- Баймухамедов Рафаэль Русланович
- Аршин Александр Дмитриевич
- Пасечный Леонид Витальевич

Преподаватель
- Петруша Полина Георгиевна

Скачаем датасет с Яндекс.Диска

In [ ]:
import requests
from urllib.parse import urlencode

base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'
public_key = "https://disk.yandex.ru/d/V1sJpR-SUJ_b8A"

final_url = base_url + urlencode(dict(public_key=public_key))
response = requests.get(final_url)
download_url = response.json()['href']

download_response = requests.get(download_url)
with open('dataset.xlsx', 'wb') as f:
    f.write(download_response.content)

Прочитаем в датафрейм наш файл

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

xlsx_path = "dataset.xlsx"
if not xlsx_path:
    raise FileNotFoundError("xlsx файл не найден")
print("Найден XLSX:", xlsx_path)

df = pd.read_excel(xlsx_path, sheet_name=0, header=[0,1])
print("Данные загружены в df")

Настроим pandas

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

# Предобработка данных

Посмотрим на датафрейм до какой-либо предобработки данных

In [ ]:
df.head()

Соединим заголовки первого и второго уровня вместе. Также уберем пробелы между словами в столбцах, заменив их на "_" и приведем названия столбцов к нижнему регистру.

In [ ]:
import re
from collections import Counter, defaultdict

def clean(s):
    if s is None: return ""
    s = str(s).replace("\n"," ").replace("\xa0"," ").strip()
    return re.sub(r"\s+"," ", s)

flat = []
for top, sub in df.columns:
    top, sub = clean(top), clean(sub)
    name = sub if (not top or top.lower().startswith("unnamed")) else f"{top}__{sub}" if sub else top
    name = name.replace("ё","е")
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[\\/:;,\"'()]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    name = name.lower()
    flat.append(name)

cnt = Counter(flat); seen = defaultdict(int); uniq = []
for n in flat:
    seen[n] += 1
    uniq.append(n if cnt[n] == 1 else f"{n}__{seen[n]}")

df.columns = uniq

Переименуем некоторые столбцы

In [ ]:
to_rename = {
    "пищевые_вещества_макро-_и_микроэлементы": "пищевые_вещества_макро_и_микроэлементы",
    "минеральные_и_минерало-органические_природные_субстанции_цеолиты_гуминовые_кислоты":"минеральные_и_минерало_органические_природные_субстанции_цеолиты_и_гуминовые_кислоты",
    "система_органов_костно-мышечная_сиситема": "система_органов_костно_мышечная_система",
    "система_органов_форма_выпуска":"форма_выпуска",
    "система_органов_продолжительность_приема":"продолжительность_приема",
    "система_органов_происхождение":"происхождение",
    "система_органов_сырье_растительное_животное_биологическое":"сырье",
    "система_органов_сердечно-сосудистая_система":"система_органов_сердечно_сосудистая_система"
}

df = df.rename(columns=to_rename)

Создадим новый столбец "рекомендации_по_применению". Информацию для них берем из столбца Этикетка, затем отчищаем оттуда взятую инфу.

In [ ]:
df['рекомендации_по_применению'] = pd.NA
dot = 0


for row in range(len(df)):
    string = str(df.at[row, 'этикетка'])
    value_1 = ""

    dot = string.find(".")+1
    second_dot_index = string.find(".", dot)

    if second_dot_index != -1:
        str_for_et = string[second_dot_index+1::].strip()
    else:
        str_for_et = string

    if "Рекомендации по применению" not in string:
        df.at[row, 'рекомендации_по_применению'] = None

    else:
        start_index = string.find("Рекомендации по применению")
        current_string = string[start_index:]
        duration_index = current_string.find("Продолжительность")

        if duration_index != -1:
            end_index = start_index + duration_index
            value_1 = string[start_index:end_index].strip()

            if value_1.endswith((':',' ','.')):
                value_1 = value_1[:-1].strip()

        else:
            index = 0
            while True:
                if index + 1 >= len(current_string):
                    value_1 = current_string.strip()
                    break

                if current_string[index+1] != ".":
                    index += 1
                else:
                    value_1 = string[start_index : start_index + index + 2].strip()
                    break

    df.at[row, 'рекомендации_по_применению'] = value_1
    df.at[row, 'этикетка'] = str_for_et

Создадим новый столбцы:
- Количество единиц на прием
- Количество приемов в день
- Суммарное количество единиц за период (Количество единиц на приеме * Количество приемов в день * Продолжительность приема)

In [ ]:
df["количество_единиц_на_прием"] = pd.NA
df["количество_приемов_в_день"] = pd.NA
df["суммарное_количество_единиц_за_период"] = pd.NA

re_po_range = re.compile(r'по\s*(\d+)\s*-\s*(\d+)', flags=re.IGNORECASE)
re_po_single = re.compile(r'по\s*(\d+)(?!\s*-\s*\d+)', flags=re.IGNORECASE)
re_grams = re.compile(r'\((\d+)\s*г\)', flags=re.IGNORECASE)
re_times_range = re.compile(r'(\d+)\s*-\s*(\d+)\s*раз', flags=re.IGNORECASE)
re_times_single = re.compile(r'(\d+)\s*раз', flags=re.IGNORECASE)
re_grams_inline = re.compile(r'\b(\d+)\s*(?:г\b|грамм\w*)', flags=re.IGNORECASE)
re_ml = re.compile(r'\b(\d+)\s*мл\b(?!\s*питьевой воды)', flags=re.IGNORECASE)


for i in range(len(df)):
    string_raw = str(df.at[i, 'рекомендации_по_применению'])
    string = string_raw.lower()
    units = np.nan
    times = np.nan

    # количество_единиц_на_прием
    m = re_po_range.search(string)
    if m:
        units = int(m.group(2))
    else:
        m = re_po_single.search(string)
        if m:
            units = int(m.group(1))
        else:
            if 'пакет' in string:
                idx = string.find('пакет')
                left_window = string[max(0, idx-12):idx]
                m = re.search(r'(\d+)\s*-\s*(\d+)', left_window)
                if m:
                    units = int(m.group(2))
                else:
                    m = re.search(r'(\d+)', left_window)
                    if m:
                        units = int(m.group(1))
            if np.isnan(units):
                m = re_grams.search(string)
                if m:
                    units = int(m.group(1))
            if np.isnan(units):
                m = re_grams_inline.search(string)
                if m:
                    units = int(m.group(1))
            if np.isnan(units):
                m = re_ml.search(string)
                if m:
                    units = int(m.group(1))
    df.at[i, 'количество_единиц_на_прием'] = units

    # количество_приемов_в_день

    times = 1
    m = re_times_range.search(string)
    if m:
        times = int(m.group(2))
    else:
        m = re_times_single.search(string)
        if m:
            times = int(m.group(1))
    df.at[i, 'количество_приемов_в_день'] = times

# Заменим некоторые неправильно обработанные строки вручную
df.loc[39, 'количество_единиц_на_прием'] = 1
df.loc[59, 'количество_единиц_на_прием'] = 2
df.loc[91, 'количество_единиц_на_прием'] = 1
df.loc[102, 'количество_единиц_на_прием'] = 6
df.loc[178, 'количество_единиц_на_прием'] = 2
df.loc[215, 'количество_единиц_на_прием'] = 2
df.loc[222, 'количество_единиц_на_прием'] = 5
df.loc[723, 'количество_единиц_на_прием'] = 4
df.loc[928, 'количество_единиц_на_прием'] = 1
df.loc[1207, 'количество_единиц_на_прием'] = 3
df.loc[1219, 'количество_единиц_на_прием'] = 1
df.loc[1262, 'количество_единиц_на_прием'] = 1
df.loc[1263, 'количество_единиц_на_прием'] = 2
df.loc[1264, 'количество_единиц_на_прием'] = 3
df.loc[1265, 'количество_единиц_на_прием'] = 3
df.loc[1289, 'количество_единиц_на_прием'] = 1
df.loc[1492, 'количество_единиц_на_прием'] = 1
df.loc[1693, 'количество_единиц_на_прием'] = 1
df.loc[1785, 'количество_единиц_на_прием'] = 1
df.loc[1786, 'количество_единиц_на_прием'] = 2
df.loc[1855, 'количество_единиц_на_прием'] = 6
df.loc[1884, 'количество_единиц_на_прием'] = 10
df.loc[1991, 'количество_единиц_на_прием'] = 2
df.loc[2117, 'количество_единиц_на_прием'] = 1
df.loc[2325, 'количество_единиц_на_прием'] = 3
df.loc[2391, 'количество_единиц_на_прием'] = 1
df.loc[2487, 'количество_единиц_на_прием'] = 6
df.loc[2517, 'количество_единиц_на_прием'] = 2
df.loc[2518, 'количество_единиц_на_прием'] = 4
df.loc[2523, 'количество_единиц_на_прием'] = 4
df.loc[2524, 'количество_единиц_на_прием'] = 2
df.loc[2531, 'количество_единиц_на_прием'] = 5
df.loc[2545, 'количество_единиц_на_прием'] = 8
df.loc[2598, 'количество_единиц_на_прием'] = 1
df.loc[2675, 'количество_единиц_на_прием'] = 3
df.loc[2724, 'количество_единиц_на_прием'] = 1
df.loc[2729, 'количество_единиц_на_прием'] = 2
df.loc[2843, 'количество_единиц_на_прием'] = 1
df.loc[2857, 'количество_единиц_на_прием'] = 3
df.loc[2859, 'количество_единиц_на_прием'] = 3
df.loc[2889, 'количество_единиц_на_прием'] = 1
df.loc[2894, 'количество_единиц_на_прием'] = 1
df.loc[2895, 'количество_единиц_на_прием'] = 1
df.loc[2900, 'количество_единиц_на_прием'] = 2
df.loc[2902, 'количество_единиц_на_прием'] = 3
df.loc[2937, 'количество_единиц_на_прием'] = 2
df.loc[2951, 'количество_единиц_на_прием'] = 1
df.loc[3028, 'количество_единиц_на_прием'] = 3
df.loc[3314, 'количество_единиц_на_прием'] = 5
df.loc[3470, 'количество_единиц_на_прием'] = 2
df.loc[3649, 'количество_единиц_на_прием'] = 1
df.loc[3660, 'количество_единиц_на_прием'] = 1
df.loc[3960, 'количество_единиц_на_прием'] = 1
df.loc[3963, 'количество_единиц_на_прием'] = 8
df.loc[3965, 'количество_единиц_на_прием'] = 3
df.loc[4026, 'количество_единиц_на_прием'] = 1
df.loc[4103, 'количество_единиц_на_прием'] = 1
df.loc[4141, 'количество_единиц_на_прием'] = 2

df.loc[22, 'количество_приемов_в_день'] = 4
df.loc[39, 'количество_приемов_в_день'] = 6
df.loc[91, 'количество_приемов_в_день'] = 2
df.loc[102, 'количество_приемов_в_день'] = 6
df.loc[319, 'количество_приемов_в_день'] = 3

Напишем функцию, выводящую суммарную информацию о датафрейме

In [ ]:
def print_info(df):
    num_rows = len(df)

    print("столбец | тип | кол-во уникальных значений | % пропущенных значений | список уникальных значений, если их меньше 6")
    for col in df.columns:
        miss = df[col].isna() | (df[col] == 0) | df[col].astype(str).str.strip().isin(["0", "0.0"])
        missing_percent = round(miss.mean() * 100, 2)
        count_of_unique_values = len(df[col].unique())
        unique_values = df[col].unique().tolist() if count_of_unique_values < 8 else ">5 unique values"
        print(f"{col} | {df[col].dtypes} | {count_of_unique_values} | {missing_percent}% | {unique_values}")

Посмотрим суммарную  информацию о датафрейме

In [ ]:
print_info(df)

Напишем функцию, которая заменяет значение из списка в строке столбца на заданное значение. Таким образом, заменим:
- орфографические ошибки
- продолжительность приёма в значение месяца по максимальному значению
- срок годности в месяцы
- возраст детей в месяцы
- столбцы с двумя уникальными значениями в бинарные

In [ ]:
def replace_exact(df, col, variants, target):
    df.loc[df[col].isin(variants), col] = target

pairs = [
    # Исправление орфографических ошибок
    ["пищевые_вещества_белки_пептиды_аминокислоты_нуклеиновые_кислоты", ["аминоксилоты"], "аминокислоты"],
    ["минорные_компоненты_растений_алкалоиды", ["алкалод", "алкалоид"], "алкалоиды"],
    ["пищевые_вещества_углеводы_и_продукты_их_переработки", ["полисахарид", "полисахарилы", "полисхариды"], "полисахариды"],
    ["минорные_компоненты_растений_гидроксикоричные_кислоты", ["гидрокор"], "гидроксикор"],
    ["минорные_компоненты_растений_ферменты", ["фермент"], "ферменты"],
    ["минеральные_и_минерало_органические_природные_субстанции_цеолиты_и_гуминовые_кислоты", ["цеолит"], "цеолиты"],
    ["система_органов_для_беременных_кормящих_и_планирующих_беременность", ["берем"], "беременные"],
    ["система_органов_костно_мышечная_система", ["суст", "суств"], "суставы"],
    ["система_органов_нервная_система", ["невр", "неврная", "нерврная", "нерв"], "нервная"],
    ["система_органов_иммунная_система", ["имм", "имммун", "иммун"], "иммунитет"],
    ["система_органов_дерматологические_бад", ["коэа"], "кожа"],
    ["система_органов_сердечно_сосудистая_система", ["серд"], "сердце"],
    ["система_органов_противопаразитарные_бад", ["паразит"], "паразиты"],
    ["система_органов_дыхательная_система", ["легк"], "легкие"],
    ["система_органов_противомикробные_бад", ["бакт", "бактер"], "бактерия"],
    ["система_органов_противомикробные_бад", ["вир"], "вирус"],
    ["система_органов_противомикробные_бад", ["вир"], "вирус"],
    ["система_органов_противомикробные_бад", ["грию"], "гриб"],

    # Переведем продолжительность приема в месяцы
    ["продолжительность_приема", ["1 месяц и менее", "постоянно"], "1"],
    ["продолжительность_приема", ["1-2 месяца"], "2"],
    ["продолжительность_приема", ["3 месяца"], "3"],
    ["продолжительность_приема", ["6 месяцев"], "6"],
    ["продолжительность_приема", ["9 месяцев"], "9"],
    ["продолжительность_приема", ["не указано"], pd.NA],

    # Переведем срок годности в месяцы
    ["срок_годности", ["1 год"], "12"],
    ["срок_годности", ["1 год, 2 месяца"], "14"],
    ["срок_годности", ["1,5 года"], "18"],
    ["срок_годности", ["15 суток"], "0.5"],
    ["срок_годности", ["1 год, 2 года", "2 года", "2 года, 1 год", "2 года, 1,5 года"], "24"],
    ["срок_годности", ["2,5 года"], "30"],
    ["срок_годности", ["3 года"], "36"],
    ["срок_годности", ["3,5 года"], "42"],
    ["срок_годности", ["4 года"], "48"],
    ["срок_годности", ["5 лет"], "60"],
    ["срок_годности", ["1 месяц"], "1"],
    ["срок_годности", ["2 месяца"], "2"],
    ["срок_годности", ["2,5 месяца"], "2.5"],
    ["срок_годности", ["3 месяца"], "3"],
    ["срок_годности", ["6 месяцев"], "6"],
    ["срок_годности", ["8 месяцев"], "8"],
    ["срок_годности", ["11 месяцев"], "11"],
    ["срок_годности", ["14 месяцев"], "14"],
    ["срок_годности", ["19 месяцев"], "19"],
    ["срок_годности", ["20 месяцев"], "20"],
    ["срок_годности", ["21 месяц"], "21"],
    ["срок_годности", ["25 месяцев"], "25"],
    ["срок_годности", ["28 месяцев"], "28"],
    ["срок_годности", ["32 месяца"], "32"],

    # Переведем возраст детей в месяцы
    ["группа_населения_возраст_детей", ["с рождения"], "0"],
    ["группа_населения_возраст_детей", ["от 3 месяцев"], "3"],
    ["группа_населения_возраст_детей", ["от 1 года"], "12"],
    ["группа_населения_возраст_детей", ["от 1,5 лет"], "24"],
    ["группа_населения_возраст_детей", ["от 3 лет"], "36"],
    ["группа_населения_возраст_детей", ["от 4 лет"], "48"],
    ["группа_населения_возраст_детей", ["от 5 лет"], "60"],
    ["группа_населения_возраст_детей", ["от 7 лет"], "84"],
    ["группа_населения_возраст_детей", ["от 11 лет"], "132"],
    ["группа_населения_возраст_детей", ["от 12 лет"], "144"],
    ["группа_населения_возраст_детей", ["от 14 лет"], "168"],

    # Преобразуем форму выпуска БАДа в новые значения
    ["форма_выпуска", ["таблетки","капсулы","пастилки","пилюли","драже","леденцы","плитки","таблетки, капсулы", "капсулы, пилюли","таблетки, капсулы, пастилки","таблетки, пилюли","капсулы, пастилки"], "твердое"],
    ["форма_выпуска", ["порошок","гранулы","порошок, гранулы"], "сыпучее"],
    ["форма_выпуска", ["гели","пасты","желе"], "полутвердое"],
    ["форма_выпуска", ["растворы","суспензия"], "жидкое"],
    ["форма_выпуска", ["сбор"], "сборы"],
    ["форма_выпуска", ["таблетки, порошок","таблетки, капсулы, порошок","капсулы, порошок","капсулы, порошок ", "таблетки, порошок, капсулы, гранулы"], "твердое, сыпучее"],
    ["форма_выпуска", ["капсулы, растворы","таблетки, капсулы, растворы","таблетки, растворы"], "твердое, жидкое"],
    ["форма_выпуска", ["таблетки, капсулы, сбор","капсулы, сбор"], "твердое, сборы"],
    ["форма_выпуска", ["капсулы, порошок, пасты"], "твердое, сыпучее, полутвердое"],
    ["форма_выпуска", ["раствор, сбор"], "жидкое, сборы"],

    # Преобразуем столбцы с двумя уникальными значениями в бинарные
    ["происхождение", ["иностранное"], "0"],
    ["происхождение", ["отечественное"], "1"],
    ["пищевые_вещества_витамины_витаминоподобные_вещества_и_коферменты", ["вит"], "1"],
    ["пищевые_вещества_макро_и_микроэлементы", ["элементы"], "1"],
    ["пищевые_вещества_белки_пептиды_аминокислоты_нуклеиновые_кислоты", ["аминокислоты"], "1"],
    ["минорные_компоненты_растений_фенольные_соединения", ["фенольн"], "1"],
    ["минорные_компоненты_растений_алкалоиды", ["алкалоиды"], "1"],
    ["пробиотики_в_монокультурах_и_ассоциациях_пробиотические_микроорганизмы", ["пробиотики"], "1"],
    ["пищевые_вещества_углеводы_и_продукты_их_переработки", ["полисахариды"], "1"],
    ["минорные_компоненты_растений_сапонины", ["сапонины"], "1"],
    ["минорные_компоненты_растений_терпеноиды", ["терпен"], "1"],
    ["минорные_компоненты_растений_естественные_метаболиты_и_стимуляторы_метаболизма", ["ест"], "1"],
    ["минорные_компоненты_растений_гидроксикоричные_кислоты", ["гидроксикор"], "1"],
    ["минорные_компоненты_растений_ферменты", ["ферменты"], "1"],
    ["минорные_компоненты_растений_дубильные_вещества", ["дуб"], "1"],
    ["минеральные_и_минерало_органические_природные_субстанции_цеолиты_и_гуминовые_кислоты", ["цеолиты"], "1"],
    ["система_органов_для_беременных_кормящих_и_планирующих_беременность", ["беременные"], "1"],
    ["система_органов_костно_мышечная_система", ["суставы"], "1"],
    ["система_органов_нервная_система", ["нервная"], "1"],
    ["система_органов_иммунная_система", ["иммунитет"], "1"],
    ["система_органов_пищеварительный_тракт_и_обмен_веществ", ["жкт"], "1"],
    ["система_органов_мочеполовая_система", ["почки"], "1"],
    ["система_органов_дерматологические_бад", ["кожа"], "1"],
    ["система_органов_органы_чувств", ["глаза"], "1"],
    ["система_органов_сердечно_сосудистая_система", ["сердце"], "1"],
    ["система_органов_противоопухолевые_бад", ["онко"], "1"],
    ["система_органов_противопаразитарные_бад", ["паразиты"], "1"],
    ["система_органов_кровь_и_система_кроветворения", ["кровь"], "1"],
    ["группа_населения_предназначен_для_детей", ["дети"], "1"],
    ["группа_населения_предназначен_для_взрослых", ["взрослые"], "1"],
    ["группа_населения_пожилые", ["пожилые"], "1"],
]

for i in range(len(pairs)):
    replace_exact(df, pairs[i][0],pairs[i][1], pairs[i][2])

Изменим тип некоторых столбцов

In [ ]:
df["продолжительность_приема"] = (pd.to_numeric(df["продолжительность_приема"], errors="coerce").astype("Int64"))
df["срок_годности"] = (pd.to_numeric(df["срок_годности"],errors="coerce").astype("Float64"))
df["группа_населения_возраст_детей"] = (pd.to_numeric(df["группа_населения_возраст_детей"], errors="coerce").astype("Int64"))
df["происхождение"] = pd.to_numeric(df["происхождение"], errors="coerce").astype("Int8")
binary_cols = [
    "пищевые_вещества_витамины_витаминоподобные_вещества_и_коферменты",
    "пищевые_вещества_макро_и_микроэлементы",
    "пищевые_вещества_белки_пептиды_аминокислоты_нуклеиновые_кислоты",
    "минорные_компоненты_растений_фенольные_соединения",
    "минорные_компоненты_растений_алкалоиды",
    "пробиотики_в_монокультурах_и_ассоциациях_пробиотические_микроорганизмы",
    "пищевые_вещества_углеводы_и_продукты_их_переработки",
    "минорные_компоненты_растений_сапонины",
    "минорные_компоненты_растений_терпеноиды",
    "минорные_компоненты_растений_естественные_метаболиты_и_стимуляторы_метаболизма",
    "минорные_компоненты_растений_гидроксикоричные_кислоты",
    "минорные_компоненты_растений_ферменты",
    "минорные_компоненты_растений_дубильные_вещества",
    "минеральные_и_минерало_органические_природные_субстанции_цеолиты_и_гуминовые_кислоты",
    "система_органов_для_беременных_кормящих_и_планирующих_беременность",
    "система_органов_костно_мышечная_система",
    "система_органов_нервная_система",
    "система_органов_иммунная_система",
    "система_органов_пищеварительный_тракт_и_обмен_веществ",
    "система_органов_мочеполовая_система",
    "система_органов_дерматологические_бад",
    "система_органов_органы_чувств",
    "система_органов_сердечно_сосудистая_система",
    "система_органов_противоопухолевые_бад",
    "система_органов_противопаразитарные_бад",
    "система_органов_кровь_и_система_кроветворения",
    "группа_населения_предназначен_для_детей",
    "группа_населения_предназначен_для_взрослых",
    "группа_населения_пожилые",
]
for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("Int8")

Посчитаем столбец "суммарное_количество_единиц_за_период"

In [ ]:
df['суммарное_количество_единиц_за_период'] = (
    df['количество_единиц_на_прием'] * df['количество_приемов_в_день'] * df['продолжительность_приема'] * 30
)

Посмотрим суммарную ифнормацию о получившемся датафрейме

In [ ]:
print_info(df)

Посмотрим как выглядит датафрейм на данный момент

In [ ]:
df.head()

Вопросы на рассмотрение:
Порог процента пустых значений, при котором мы отбросим столбец

# Сохранение изменений

In [ ]:
!pip -q install jupytext nbstripout

from google.colab import drive
drive.mount('/content/drive')

NOTEBOOK = "/content/drive/MyDrive/Colab Notebooks/3311_bajmuhamedov_arshin_pasechny_practice_bad.ipynb"

cfg = '''formats = "ipynb,py:percent"
cell_metadata_filter = "-all"
notebook_metadata_filter = "kernelspec,jupytext"
'''
with open("/content/.jupytext.toml", "w", encoding="utf-8") as f:
    f.write(cfg)

import os, pathlib, time, textwrap, subprocess, json
ipynb_path = pathlib.Path(NOTEBOOK)
py_path = ipynb_path.with_suffix(".py")

if not ipynb_path.exists():
    raise FileNotFoundError(f"Не найден .ipynb: {ipynb_path}")

print("IPYNB:", ipynb_path)
print("PY:", py_path)

!nbstripout "{NOTEBOOK}"

if py_path.exists():
    py_path.unlink()
!jupytext --to py:percent "{NOTEBOOK}"

import datetime
stat = py_path.stat()
print("\nОбновлён .py:", py_path)